嵌入模型概述
Text Embedding Models：文档嵌入模型，提供将文本编码为向量的能力，即 文档向量化 。 文档写入
和 用户查询匹配 前都会先执行文档嵌入编码，即向量化。

文档嵌入模型（Text Embedding Models）负责将 文本 转换为 向量表示 ，即模型赋予了文本计算机可
理解的数值表示，使文本可用于向量空间中的各种运算，大大拓展了文本分析的可能性，是自然语言处
理领域非常重要的技术。

实现原理：通过 特定算法 （如Word2Vec）将语义信息编码为固定维度的向量，具体算法细节需后
续深入。
关键特性：相似的词在向量空间中距离相近，例如"猫"和"犬"的向量夹角小于"猫"和"汽车"。

文本嵌入为 LangChain 中的问答、检索、推荐等功能提供了重要支持。具体为：

语义匹配 ：通过计算两个文本的向量余弦相似度，判断它们在语义上的相似程度，实现语义匹配。
文本检索 ：通过计算不同文本之间的向量相似度，可以实现语义搜索，找到向量空间中最相似的文本。
信息推荐 ：根据用户的历史记录或兴趣嵌入生成用户向量，计算不同信息的向量与用户向量的相似度，推荐相似的信息。
知识挖掘 ：可以通过聚类、降维等手段分析文本向量的分布，发现文本之间的潜在关联，挖掘知识。
自然语言处理 ：将词语、句子等表示为稠密向量，为神经网络等下游任务提供输入。

1、句子的向量化

举例：

In [1]:
from langchain_openai import OpenAIEmbeddings
import os
import dotenv

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 这里获取嵌入模型
embedding_model = OpenAIEmbeddings(
    # model="text-embedding-ada-002"
    model = "text-embedding-3-large"
)

text = "Nice to meet you!"

# 句子向量化 返回：list[float]，向量
embed_query = embedding_model.embed_query(text = text,)

print(len(embed_query))  # 1536 --> 3072

print(embed_query[:10])

3072
[-0.027939368039369583, 0.03950421139597893, -0.020670808851718903, -0.0008068201714195311, 0.015577414073050022, -0.00793732050806284, -0.013834580779075623, 0.01602325402200222, 0.015712516382336617, 0.06263390183448792]


2、文档的向量化

文档的向量化，接收的参数是字符串数组。

举例1：


In [4]:
from langchain_openai import OpenAIEmbeddings
import numpy as np
import pandas as pd
import os
import dotenv

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 初始化嵌入模型
embeddings_model = OpenAIEmbeddings(model="text-embedding-ada-002")

# 待嵌入的文本列表
texts = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!"
]

# 生成嵌入向量
#  list[list[float]]
embeddings = embeddings_model.embed_documents(texts)


for i in range(len(texts)):
    print(f"{texts[i]}:{embeddings[i][:3]}",end="\n\n")


Hi there!:[-0.020325319841504097, -0.007096723187714815, -0.022839006036520004]

Oh, hello!:[0.004446744918823242, -0.014353534206748009, 0.0019785689655691385]

What's your name?:[-0.004887457471340895, -0.009618516080081463, 0.007236444391310215]

My friends call me World:[-0.004588752053678036, -0.014497518539428711, 0.01022490207105875]

Hello World!:[0.002393412170931697, 0.0002737858740147203, -0.0023398753255605698]



举例2：


In [5]:
from dotenv import load_dotenv
from langchain_community.document_loaders import CSVLoader
from langchain_openai import OpenAIEmbeddings


embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
)

# 情况1：
loader = CSVLoader("./asset/load/03-load.csv", encoding="utf-8")
# 一边加载一边切分 :底层源码使用递归字符拆分 RecursiveCharacterTextSplitter
'''
        if text_splitter is None:
            if not _HAS_TEXT_SPLITTERS:
                msg = (
                    "Unable to import from langchain_text_splitters. Please specify "
                    "text_splitter or install langchain_text_splitters with "
                    "`pip install -U langchain-text-splitters`."
                )
                raise ImportError(msg)

            text_splitter_: TextSplitter = RecursiveCharacterTextSplitter()
        else:
            text_splitter_ = text_splitter
        docs = self.load()
        return text_splitter_.split_documents(docs)
'''
docs = loader.load_and_split()

#print(len(docs))

# 存放的是每一个chrunk的embedding。
embeded_docs = embeddings_model.embed_documents([doc.page_content for doc in docs])
print(len(embeded_docs))
# 表示的是每一个chrunk的embedding的维度
print(len(embeded_docs[0]))
print(embeded_docs[0][:10])

4
3072
[0.0011534227523952723, 0.005336394999176264, -0.008949915878474712, 0.030998842790722847, -0.0017970810877159238, 0.011092216707766056, 0.021087471395730972, 0.048524416983127594, 0.0002677876618690789, -0.00012219828204251826]
